In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. STEP ONE: Create our Movie Feature Matrix
movie_data = {
    'Movie_Title': ['Die Hard', 'John Wick', 'The Hangover', 'Superbad', 'Avatar'],
    'Action_Score': [5.0, 5.0, 0.5, 1.0, 4.5],
    'Comedy_Score': [1.0, 0.2, 5.0, 5.0, 1.0],
    'SciFi_Score':  [0.0, 0.0, 0.0, 0.5, 5.0]
}
df_movies = pd.DataFrame(movie_data)

# 2. STEP TWO: Extract pure numbers for the math engine (Dropping the text column)
# We set the Movie Title as the index so it doesn't interfere with calculations
features = df_movies.set_index('Movie_Title')

# 3. STEP THREE: Calculate Cosine Similarity Matrix
# This returns a grid comparing every movie to every other movie
similarity_matrix = cosine_similarity(features)

# Convert it into a clean, readable DataFrame with titles as headers
df_similarity = pd.DataFrame(similarity_matrix, index=features.index, columns=features.index)

print("--- Movie Feature Matrix ---")
display(df_movies)

print("\n--- Calculated Cosine Similarity Grid ---")
display(df_similarity.round(2))

# 4. STEP FOUR: Create the Recommendation Engine Function
def get_recommendations(target_movie):
    print(f"\nBecause you watched '{target_movie}', we recommend:")
    # Grab the similarity column for our target movie, sort from high to low, and skip the 1st one (itself)
    recommendations = df_similarity[target_movie].sort_values(ascending=False)
    return recommendations.iloc[1:3] # Get top 2 nearest matches

# Test our engine!
print(get_recommendations('Die Hard'))

--- Movie Feature Matrix ---


,Movie_Title,Action_Score,Comedy_Score,SciFi_Score
0,Die Hard,5.0,1.0,0.0
1,John Wick,5.0,0.2,0.0
2,The Hangover,0.5,5.0,0.0
3,Superbad,1.0,5.0,0.5
4,Avatar,4.5,1.0,5.0



--- Calculated Cosine Similarity Grid ---


Movie_Title,Die Hard,John Wick,The Hangover,Superbad,Avatar
Movie_Title,,,,,
Die Hard,1.00,0.99,0.29,0.38,0.68
John Wick,0.99,1.00,0.14,0.23,0.67
The Hangover,0.29,0.14,1.00,0.99,0.21
Superbad,0.38,0.23,0.99,1.00,0.34
Avatar,0.68,0.67,0.21,0.34,1.00



Because you watched 'Die Hard', we recommend:
Movie_Title
John Wick    0.987636
Avatar       0.677681
Name: Die Hard, dtype: float64


In [1]:
import sqlite3
import pandas as pd

# 1. Open a live connection to our system database
conn = sqlite3.connect('production_movies.db')
cursor = conn.cursor()

# 2. Enable Foreign Key constraints (ensures data integrity)
cursor.execute("PRAGMA foreign_keys = ON;")

# 3. Create the Movies Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Movies (
    movie_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    action_score REAL,
    comedy_score REAL,
    scifi_score REAL
)
''')

# 4. Create the Watch_History Table (Linked to Movies via movie_id)
cursor.execute('''
CREATE TABLE IF NOT EXISTS Watch_History (
    history_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_name TEXT NOT NULL,
    movie_id INTEGER,
    user_rating REAL,
    FOREIGN KEY (movie_id) REFERENCES Movies(movie_id)
)
''')

# Clear old data if running multiple times so we don't get duplicates
cursor.execute("DELETE FROM Watch_History")
cursor.execute("DELETE FROM Movies")

# 5. Insert Data into Movies Table
movies_to_insert = [
    ('Die Hard', 5.0, 1.0, 0.0),
    ('John Wick', 5.0, 0.2, 0.0),
    ('The Hangover', 0.5, 5.0, 0.0),
    ('Superbad', 1.0, 5.0, 0.5),
    ('Avatar', 4.5, 1.0, 5.0)
]
cursor.executemany("INSERT INTO Movies (title, action_score, comedy_score, scifi_score) VALUES (?, ?, ?, ?)", movies_to_insert)

# 6. Insert Data into Watch_History (Let's say user 'Darsh' watched Movie ID 1 and Movie ID 2)
user_history_to_insert = [
    ('Darsh', 1, 4.8),  # Watched Die Hard, rated 4.8
    ('Darsh', 2, 4.5),  # Watched John Wick, rated 4.5
    ('Jeet', 3, 5.0)    # Watched The Hangover, rated 5.0
]
cursor.executemany("INSERT INTO Watch_History (user_name, movie_id, user_rating) VALUES (?, ?, ?)", user_history_to_insert)
conn.commit()

# 7. THE MASTER QUERY: Join both tables to see what 'Darsh' watched and the vector profiles!
sql_join_query = '''
SELECT
    Watch_History.user_name,
    Movies.title,
    Watch_History.user_rating,
    Movies.action_score,
    Movies.comedy_score,
    Movies.scifi_score
FROM Watch_History
INNER JOIN Movies ON Watch_History.movie_id = Movies.movie_id
WHERE Watch_History.user_name = 'Darsh';
'''

# Read the SQL Join output directly into a Pandas DataFrame
df_darsh_history = pd.read_sql_query(sql_join_query, conn)

print("--- Live Relational Database Extraction for User: Darsh ---")
display(df_darsh_history)

conn.close()

--- Live Relational Database Extraction for User: Darsh ---


,user_name,title,user_rating,action_score,comedy_score,scifi_score
0,Darsh,Die Hard,4.8,5.0,1.0,0.0
1,Darsh,John Wick,4.5,5.0,0.2,0.0


In [2]:
import sqlite3
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1. Reconnect to our database file
conn = sqlite3.connect('production_movies.db')

# 2. Extract ALL master movies into a DataFrame for our AI to choose from
df_all_movies = pd.read_sql_query("SELECT * FROM Movies", conn)
features_all = df_all_movies.set_index('title').drop(columns=['movie_id'])

# 3. Extract just Darsh's highly-rated history (Rating > 4.0) to understand his taste
sql_user_taste = '''
SELECT Movies.action_score, Movies.comedy_score, Movies.scifi_score
FROM Watch_History
INNER JOIN Movies ON Watch_History.movie_id = Movies.movie_id
WHERE Watch_History.user_name = 'Darsh' AND Watch_History.user_rating >= 4.0;
'''
df_darsh_taste = pd.read_sql_query(sql_user_taste, conn)

# 4. MATH CORE: Calculate Darsh's "Taste Vector" by taking the average of his watched movies
darsh_taste_vector = df_darsh_taste.mean().values.reshape(1, -1)

# 5. VECTOR MATCHING: Compare Darsh's taste profile against ALL movies in the database
similarity_scores = cosine_similarity(darsh_taste_vector, features_all)

# 6. Package results and display top recommendations
df_all_movies['Match_Score'] = similarity_scores[0]
recommendations = df_all_movies.sort_values(ascending=False, by='Match_Score')

print("--- Darsh's Calculated Taste Profile Vector (Action, Comedy, SciFi) ---")
print(darsh_taste_vector.round(2))

print("\n--- Smart Recommendations Sorted by Your Taste Profile ---")
display(recommendations[['title', 'Match_Score']].round(2))

conn.close()

--- Darsh's Calculated Taste Profile Vector (Action, Comedy, SciFi) ---
[[5.  0.6 0. ]]

--- Smart Recommendations Sorted by Your Taste Profile ---


,title,Match_Score
0,Die Hard,1.00
1,John Wick,1.00
4,Avatar,0.67
3,Superbad,0.31
2,The Hangover,0.22
